# PXIe-4080 Raw Measurement Error — 3D Analysis

Loads the same TDMS data files as the gain-error notebook, but keeps every
individual test-point measurement rather than aggregating to a single gain-error
per sweep.  For each sample:

```
error_ppm = (dut_measurement - source_voltage) / range_voltage * 1_000_000
```

Exports a single self-contained HTML file (`meas_error_3d.html`) with three
interactive Plotly 3D scatter plots:

1. **All data** — 10 V and 300 V ranges combined
2. **10 V range only**
3. **300 V range only**

Each plot has one trace per serial number (per range for plot 1).  Use the legend
to show/hide individual units.  Use the toolbar buttons to toggle hover or reset
the 3D camera.

## 1. Imports and filename parser

In [ ]:
import os
import re
import base64
from datetime import datetime

import pandas as pd
from nptdms import TdmsFile
import plotly.graph_objects as go
import plotly.colors as pc


def parse_filename(filename):
    """Parse TDMS filename into (serial, temp_set, rh_set, datetime).

    Expected pattern (spaces normalised):
      <SERIAL>_Temp_Set_<TEMP>_RH_Set_<RH>_MM_DD_YYYY_HH_MM_SS_AM|PM.tdms
    """
    name = filename[:-5].strip()
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'\s+(AM|PM)$', r'_\1', name)
    name = name.replace('RH_Set_ ', 'RH_Set_')

    pattern = re.compile(
        r'^(?P<serial>[^_]+)_Temp_Set_(?P<temp>[^_]+)_RH_Set_(?P<rh>\d+)_'
        r'(?P<month>\d+)_(?P<day>\d+)_(?P<year>\d+)_'
        r'(?P<hour>\d+)_(?P<minute>\d+)_(?P<second>\d+)_(?P<am_pm>AM|PM)$'
    )
    match = pattern.match(name)
    if not match:
        raise ValueError(f'Unable to parse filename: {filename}')

    serial   = match.group('serial')
    temp_set = match.group('temp')
    rh_set   = match.group('rh')
    month    = int(match.group('month'))
    day      = int(match.group('day'))
    year     = int(match.group('year'))
    hour     = int(match.group('hour'))
    minute   = int(match.group('minute'))
    second   = int(match.group('second'))
    am_pm    = match.group('am_pm')

    dt = datetime(year, month, day, hour, minute, second)
    if am_pm == 'PM' and hour != 12:
        dt = dt.replace(hour=hour + 12)
    elif am_pm == 'AM' and hour == 12:
        dt = dt.replace(hour=0)
    return serial, temp_set, rh_set, dt

## 2. Helper functions

In [ ]:
def serial_sort_key(serial):
    """Sort hex serial numbers in ascending numeric order."""
    normalized = str(serial).strip()
    if normalized.lower().startswith('0x'):
        normalized = normalized[2:]
    try:
        return (0, int(normalized, 16), str(serial))
    except ValueError:
        return (1, str(serial))


def get_first_available_channel(group, channel_names):
    """Return data from the first channel name that exists in *group*."""
    for channel_name in channel_names:
        if channel_name in group:
            return group[channel_name][:]
    joined = ', '.join(channel_names)
    raise KeyError(f'No channel found from candidates: {joined}')


def inject_branding_header(html_file_path, logo_base64_data):
    """Insert the NI/Emerson logo and NI Confidential banner immediately after <body>."""
    with open(html_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    # Remove any previously injected header block.
    html_content = re.sub(
        r'<!-- NI_CONFIDENTIAL_HEADER_START -->[\s\S]*?<!-- NI_CONFIDENTIAL_HEADER_END -->\s*',
        '',
        html_content,
        flags=re.IGNORECASE,
    )

    branding_html = f"""
<!-- NI_CONFIDENTIAL_HEADER_START -->
<div id="ni-confidential-header" style="padding: 16px 24px 14px; border-bottom: 2px solid #d0d0d0; margin-bottom: 14px; background: #fff;">
  <div style="display: flex; align-items: flex-start; justify-content: space-between;">
    <img src="data:image/jpeg;base64,{logo_base64_data}" alt="NI/Emerson Logo" style="height: 58px; width: auto; object-fit: contain;">
    <div style="flex: 1; text-align: center; margin-right: 58px;">
      <h1 style="color: #c62828; font-size: 24px; margin: 0 0 8px 0; font-weight: 700;">NI Confidential</h1>
      <p style="color: #111; font-size: 14px; margin: 0; line-height: 1.35;">
        This document contains proprietary information of NI/Emerson and is intended for internal use only. Unauthorized distribution is prohibited.
      </p>
    </div>
  </div>
</div>
<!-- NI_CONFIDENTIAL_HEADER_END -->
"""

    body_match = re.search(r'<body[^>]*>', html_content, flags=re.IGNORECASE)
    if body_match:
        insert_at = body_match.end()
        html_content = html_content[:insert_at] + '\n' + branding_html + '\n' + html_content[insert_at:]
    else:
        html_content = branding_html + '\n' + html_content

    with open(html_file_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

## 3. Configuration

Set `ROOT_PATH` to the folder that contains one sub-folder per DUT serial number.
Each sub-folder should hold the `.tdms` files for that unit.

In [ ]:
ROOT_PATH  = r'C:\temp\EnvironmentalTesting\Round2Data'
OUTPUT_HTML = 'meas_error_3d.html'

# Divisor for PPM conversion: error_ppm = (dut - src) / divisor * 1_000_000
RANGE_DIVISORS = {'10V': 10.0, '300V': 300.0}

# ---- logo ----
LOGO_PATH = 'NI-Emerson.jpg'
with open(LOGO_PATH, 'rb') as f:
    LOGO_BASE64 = base64.b64encode(f.read()).decode('utf-8')
print(f'Logo loaded and encoded ({len(LOGO_BASE64)} bytes base64 data)')

## 4. Load TDMS data

Walks each serial sub-folder, reads the `Test Points` and `DUT Measurements`
channels for the 10 V and 300 V ranges, and expands every sweep into individual
per-test-point rows.  The measurement error is converted to PPM of range.

In [ ]:
data = []

if not os.path.exists(ROOT_PATH):
    raise FileNotFoundError(f'Path not found: {ROOT_PATH}  — please update ROOT_PATH.')

# Collect every .tdms file up front so we can report progress while loading.
tdms_files = []
for subdir in os.listdir(ROOT_PATH):
    serial_path = os.path.join(ROOT_PATH, subdir)
    if not os.path.isdir(serial_path):
        continue
    for file in os.listdir(serial_path):
        if file.endswith('.tdms'):
            tdms_files.append((serial_path, file))

total_files = len(tdms_files)
print(f'Found {total_files} TDMS file(s) to process.')

for idx, (serial_path, file) in enumerate(tdms_files, start=1):
    print(f'\r[{idx}/{total_files}] Loading {file[:60]}', end='', flush=True)
    filepath = os.path.join(serial_path, file)
    try:
        parsed_serial, temp_set, rh_set, timestamp = parse_filename(file)
        tdms_file = TdmsFile.read(filepath)
        group = tdms_file.groups()[0]

        ranges_config = [
            ('10V',  ['Test Points 10V'],               ['DUT Measurements 10V']),
            ('300V', ['Test Points 300V', 'Test Points'],['DUT Measurements 300V', 'DUT Measurements']),
        ]

        for range_name, test_candidates, dut_candidates in ranges_config:
            test_col = next((c for c in test_candidates if c in group), None)
            dut_col  = next((c for c in dut_candidates  if c in group), None)
            if test_col is None or dut_col is None:
                continue

            test_pts = group[test_col][:]
            dut_meas = group[dut_col][:]
            divisor  = RANGE_DIVISORS[range_name]

            for src, dut in zip(test_pts, dut_meas):
                error_ppm = (float(dut) - float(src)) / divisor * 1_000_000
                data.append({
                    'timestamp':      timestamp,
                    'serial':         parsed_serial,
                    'range':          range_name,
                    'source_voltage': float(src),
                    'meas_error_ppm': error_ppm,
                })

    except Exception as e:
        print(f'\nError processing {filepath}: {e}')

print(f'\nDone. Loaded {len(data):,} sample(s) from {total_files} file(s).')

df = pd.DataFrame(data)
if df.empty:
    raise ValueError('No data loaded — check ROOT_PATH and TDMS file names.')

df = df.sort_values(['serial', 'timestamp', 'source_voltage']).reset_index(drop=True)

summary = df.groupby(['serial', 'range']).agg(
    sweeps=('timestamp', 'nunique'),
    samples=('meas_error_ppm', 'count')
)
print('\nData summary (sweeps = unique timestamps):')
print(summary.to_string())

## 5. Build 3D plots and export HTML

Three interactive Plotly 3D scatter plots are assembled and exported to a single
self-contained HTML file.

* **X axis** — sweep date / time
* **Y axis** — source voltage (V)
* **Z axis** — measurement error (PPM of range)

Each sweep appears as a vertical "curtain" of points showing the full error
profile across all test-point voltages at that moment in time.

In [ ]:
if df.empty:
    raise ValueError('No data loaded. Run the data-loading cell first.')

# â”€â”€ Colour palette: one consistent colour per serial across all three plots â”€â”€
serials = sorted(df['serial'].unique(), key=serial_sort_key)
palette = pc.qualitative.Plotly + pc.qualitative.D3 + pc.qualitative.G10
serial_colors = {ser: palette[idx % len(palette)] for idx, ser in enumerate(serials)}

# â”€â”€ Shared 3D scene axis titles â”€â”€
SCENE_LAYOUT = dict(
    xaxis=dict(title='Sweep Date / Time'),
    yaxis=dict(title='Source Voltage (V)'),
    zaxis=dict(title='Measurement Error (PPM)'),
)


def make_3d_buttons():
    """Toolbar buttons: hover toggle and reset-view for a 3D figure."""
    return [
        dict(
            type='buttons',
            direction='left',
            buttons=[
                dict(
                    label='Hover: On',
                    method='relayout',
                    args=[{'hovermode': 'closest'}],
                ),
                dict(
                    label='Hover: Off',
                    method='relayout',
                    args=[{'hovermode': False}],
                ),
                dict(
                    label='Reset View',
                    method='relayout',
                    args=[{'scene.camera.eye': {'x': 1.25, 'y': 1.25, 'z': 1.25}}],
                ),
            ],
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.0,
            xanchor='left',
            y=1.05,
            yanchor='top',
        )
    ]


def make_trace(sub_df, trace_name, color):
    """Return a go.Scatter3d trace for *sub_df* rows."""
    return go.Scatter3d(
        x=sub_df['timestamp'],
        y=sub_df['source_voltage'],
        z=sub_df['meas_error_ppm'],
        mode='markers',
        name=trace_name,
        marker=dict(size=2, color=color, opacity=0.7),
        hovertemplate=(
            '<b>' + trace_name + '</b><br>'
            'Time: %{x}<br>'
            'Source: %{y:.4f} V<br>'
            'Error: %{z:.3f} ppm<extra></extra>'
        ),
    )


# â”€â”€ Figure 1: All data (10V + 300V combined) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig_all = go.Figure()
for ser in serials:
    ser_df = df[df['serial'] == ser]
    for range_name in ('10V', '300V'):
        rd = ser_df[ser_df['range'] == range_name]
        if rd.empty:
            continue
        fig_all.add_trace(make_trace(rd, f'{ser} {range_name}', serial_colors[ser]))

fig_all.update_layout(
    title='PXIe-4080 Measurement Error — All Ranges (10V + 300V)',
    autosize=True,
    hovermode='closest',
    scene=SCENE_LAYOUT,
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.6)'),
    updatemenus=make_3d_buttons(),
)

# â”€â”€ Figure 2: 10V range only â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig_10v = go.Figure()
for ser in serials:
    sd = df[(df['serial'] == ser) & (df['range'] == '10V')]
    if sd.empty:
        continue
    fig_10v.add_trace(make_trace(sd, ser, serial_colors[ser]))

fig_10v.update_layout(
    title='PXIe-4080 Measurement Error — 10V Range',
    autosize=True,
    hovermode='closest',
    scene=SCENE_LAYOUT,
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.6)'),
    updatemenus=make_3d_buttons(),
)

# â”€â”€ Figure 3: 300V range only â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig_300v = go.Figure()
for ser in serials:
    sd = df[(df['serial'] == ser) & (df['range'] == '300V')]
    if sd.empty:
        continue
    fig_300v.add_trace(make_trace(sd, ser, serial_colors[ser]))

fig_300v.update_layout(
    title='PXIe-4080 Measurement Error — 300V Range',
    autosize=True,
    hovermode='closest',
    scene=SCENE_LAYOUT,
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.6)'),
    updatemenus=make_3d_buttons(),
)

# â”€â”€ Render each figure to an HTML div â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# First figure embeds the full plotly.js library; subsequent ones reuse it.
div_all  = fig_all.to_html(
    full_html=False, include_plotlyjs=True, div_id='plot_all',
    default_width='100%', default_height='700px',
    config={'responsive': True},
)
div_10v  = fig_10v.to_html(
    full_html=False, include_plotlyjs=False, div_id='plot_10v',
    default_width='100%', default_height='700px',
    config={'responsive': True},
)
div_300v = fig_300v.to_html(
    full_html=False, include_plotlyjs=False, div_id='plot_300v',
    default_width='100%', default_height='700px',
    config={'responsive': True},
)

# â”€â”€ Assemble combined HTML page â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Triple-single-quoted strings: single quotes used for HTML attributes so that
# no escaping is needed and the source is easy to read.
HEADER_HTML = '''<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>PXIe-4080 Measurement Error 3D Analysis</title>
  <style>
    body {font-family: Arial, Helvetica, sans-serif; margin: 0; padding: 0; background: #f0f0f0;}
    .plot-section {
      background: #ffffff;
      margin: 14px 16px;
      padding: 14px 16px 8px;
      border-radius: 4px;
      box-shadow: 0 1px 4px rgba(0,0,0,0.12);
    }
    .plot-section h2 {color: #1a1a1a; font-size: 16px; margin: 0 0 8px 0; font-weight: 600;}
  </style>
</head>
<body>
  <div class='plot-section'>
    <h2>All Ranges &#8212; 10V and 300V Measurement Error vs. Time and Source Voltage</h2>
'''

MID1_HTML = '''
  </div>
  <div class='plot-section'>
    <h2>10V Range &#8212; Measurement Error vs. Time and Source Voltage</h2>
'''

MID2_HTML = '''
  </div>
  <div class='plot-section'>
    <h2>300V Range &#8212; Measurement Error vs. Time and Source Voltage</h2>
'''

FOOTER_HTML = '''
  </div>
</body>
</html>
'''

html_page = HEADER_HTML + div_all + MID1_HTML + div_10v + MID2_HTML + div_300v + FOOTER_HTML

with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html_page)

inject_branding_header(OUTPUT_HTML, LOGO_BASE64)

print(f'HTML written to {OUTPUT_HTML}')
print(f'  {len(data):,} samples  |  {len(serials)} serial(s)  |  {len(tdms_files)} TDMS file(s)')

fig_all.show()